In [4]:
import pandas as pd
import os

def preprocess_data(df):
    """Standardize column names and prepare the 'year' column."""
    df.columns = df.columns.str.lower()

    if 'yearmonth' in df.columns:
        ### Extract year from YYYYMM format
        df['year'] = (df['yearmonth'] // 100).astype(int)

    return df

def get_explosion(df):
    """
    Recursively explodes a BOM (Bill of Materials) structure.
    Optimized by using a dictionary lookup for O(1) retrieval of components.
    """
    ### Pre-group into a lookup dictionary
    # Key: (Produced Material, Year, Plant) -> Value: List of components
    lookup = df.groupby(['produced_material', 'year', 'plant_id']).apply(
        lambda x: x[['component_material', 'component_material_release_type']].to_dict('records'),
        include_groups=False
    ).to_dict()

    final_rows = []

    ### Keep track of visited items in the current path to prevent infinite loops
    visited_path = set()

    def find_components_recursive(fin_material_id, target_material, year, plant, depth=0):
        ### Max depth or Circular Dependency
        path_key = (target_material, year, plant)
        if depth > 20 or path_key in visited_path:
            return

        visited_path.add(path_key)

        ### Fetch components from the lookup dictionary
        components = lookup.get(path_key, [])

        for comp in components:
            comp_id = comp['component_material']
            comp_type = comp['component_material_release_type']

            final_rows.append({
                'plant_id': plant,
                'fin_material_id': fin_material_id,
                'component_id': comp_id,
                'component_type': comp_type,
                'year': year,
                'level': depth + 1
            })

            ### Recurse if the component is an intermediate produced sub-assembly
            if comp_type == 'PROD':
                find_components_recursive(
                    fin_material_id,
                    comp_id,
                    year,
                    plant,
                    depth + 1
                )

        # Remove from path after dependencies are processed
        visited_path.remove(path_key)

    ### Identify starting points FIN (Finished Goods)
    # Ddrop duplicates to avoid exploding the same FIN item multiple times
    # if it appears in multiple rows in the raw BOM
    fin_root = df[df['produced_material_release_type'] == 'FIN'][
        ['produced_material', 'year', 'plant_id']
    ].drop_duplicates()

    ### Iterate through FIN items to start the explosion
    for row in fin_root.itertuples(index=False):
        find_components_recursive(
            row.produced_material, # Original FIN ID
            row.produced_material, # Starting material for recursion
            row.year,
            row.plant_id
        )

    return pd.DataFrame(final_rows)

### EXECUTION
data_path = 'task_2_data.csv'

if os.path.exists(data_path):
    ### Load and Preprocess
    df = pd.read_csv(data_path)
    df = preprocess_data(df)

    ### Run Explosion
    print("Starting Hierarchy Explosion...")
    hierarchy_df = get_explosion(df)

    ### Output results
    if not hierarchy_df.empty:
        print(f"Explosion Complete. Total relations found: {len(hierarchy_df)}")
        print(hierarchy_df.head(10))

        ### Optional: Save to CSV
        ### hierarchy_df.to_csv('exploded_bill_of_materials.csv', index=False)
    else:
        print("Explosion finished, but no components were found. Check your 'FIN' labels.")
else:
    print(f"Error: The file at {data_path} was not found.")

Starting Hierarchy Explosion...
Explosion Complete. Total relations found: 5447640
  plant_id  fin_material_id  component_id component_type  year  level
0   RLT_10            10000         50000           PROD  2024      1
1   RLT_10            10000         80070           PROD  2024      2
2   RLT_10            10000         80010           PROD  2024      3
3   RLT_10            10000         80000           PROD  2024      4
4   RLT_10            10000         70000             RM  2024      5
5   RLT_10            10000         90005            ADD  2024      5
6   RLT_10            10000         70000             RM  2024      5
7   RLT_10            10000         90005            ADD  2024      5
8   RLT_10            10000         70000             RM  2024      5
9   RLT_10            10000         90005            ADD  2024      5
